# Lesson 3: Enable Logging

> Note: In 2026, we updated the models used in this course because some older models were deprecated.
  * Model shown in the video: `titan-text-express-v1` (deprecated)
  * Model currently used in the notebooks: `nova-lite-v1`

In the videos, you will see code that uses the old model. However, we are doing the exact same exercises and concepts—only the model name and a few lines related to inference have changed.

**All notebooks** provided for this course are already updated, so please follow the notebooks when coding.
You may also notice slightly different outputs, which is expected when using a newer model.

### Import all needed packages

In [1]:
import boto3
import json
import os

bedrock = boto3.client('bedrock', region_name="us-west-2")

In [2]:
from helpers.CloudWatchHelper import CloudWatch_Helper
cloudwatch = CloudWatch_Helper()

In [3]:
log_group_name = '/my/amazon/bedrock/logs'

In [4]:
cloudwatch.create_log_group(log_group_name)

Log group '/my/amazon/bedrock/logs' created successfully.


In [5]:
loggingConfig = {
    'cloudWatchConfig': {
        'logGroupName': log_group_name,
        'roleArn': os.environ['LOGGINGROLEARN'],
        'largeDataDeliveryS3Config': {
            'bucketName': os.environ['LOGGINGBUCKETNAME'],
            'keyPrefix': 'amazon_bedrock_large_data_delivery',
        }
    },
    's3Config': {
        'bucketName': os.environ['LOGGINGBUCKETNAME'],
        'keyPrefix': 'amazon_bedrock_logs',
    },
    'textDataDeliveryEnabled': True,
}

In [6]:
bedrock.put_model_invocation_logging_configuration(loggingConfig=loggingConfig)

{'ResponseMetadata': {'RequestId': 'a26362bc-2e9d-4a22-b2a3-59ca71212eeb',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 22 Mar 2026 19:39:22 GMT',
   'content-type': 'application/json',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'a26362bc-2e9d-4a22-b2a3-59ca71212eeb'},
  'RetryAttempts': 0}}

In [7]:
bedrock.get_model_invocation_logging_configuration()

{'ResponseMetadata': {'RequestId': 'e1e23126-6ea1-40f6-8793-9a7c99376ad2',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 22 Mar 2026 19:39:23 GMT',
   'content-type': 'application/json',
   'content-length': '636',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'e1e23126-6ea1-40f6-8793-9a7c99376ad2'},
  'RetryAttempts': 0},
 'loggingConfig': {'cloudWatchConfig': {'logGroupName': '/my/amazon/bedrock/logs',
   'roleArn': 'arn:aws:iam::258622814741:role/c99355a2566044l14326101t1w2586228147-LoggingIAMRole-hNAHgDYmtta9',
   'largeDataDeliveryS3Config': {'bucketName': 'c99355a2566044l14326101t1w25862281-loggings3bucket-cuhy2yw95zkh',
    'keyPrefix': 'amazon_bedrock_large_data_delivery'}},
  's3Config': {'bucketName': 'c99355a2566044l14326101t1w25862281-loggings3bucket-cuhy2yw95zkh',
   'keyPrefix': 'amazon_bedrock_logs'},
  'textDataDeliveryEnabled': True,
  'imageDataDeliveryEnabled': True,
  'embeddingDataDeliveryEnabled': True}}

In [8]:
bedrock_runtime = boto3.client('bedrock-runtime', region_name="us-west-2")

In [9]:
prompt = "Write an article about the fictional planet Foobar."

kwargs = {
    # model updated
    "modelId": "us.amazon.nova-lite-v1:0",
    "contentType": "application/json",
    "accept": "*/*",
    "body": json.dumps(
        {   # updated due to the use of a newer model
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {
                "maxTokens": 512,
                "temperature": 0.7,
                "topP": 0.9
            }
        }
    )
}

response = bedrock_runtime.invoke_model(**kwargs)
response_body = json.loads(response.get('body').read())

content_list = response_body["output"]["message"]["content"]
text_block = next((item for item in content_list if "text" in item), None)
generation = text_block["text"] if text_block else ""

print(generation)

# Exploring the Enigmatic Planet Foobar

## Introduction

Nestled in the uncharted territories of the Andromeda Galaxy, Foobar is a planet of unparalleled beauty and mystery. Unlike any other known celestial body, Foobar captivates the imaginations of astronomers, scientists, and adventurers alike. Its vibrant landscapes, exotic wildlife, and the enigmatic culture of its inhabitants make it a subject of fascination and exploration.

## Geographical Marvels

### Diverse Terrains

Foobar is a planet of striking contrasts. Its surface is adorned with vast, lush forests, towering crystalline mountains, and expansive oceans that shimmer with an iridescent glow. The most notable geographical feature is the "Elysian Canopy," a colossal forest that stretches across thousands of miles. The trees here are known to reach heights of over 1,000 feet, their leaves reflecting a spectrum of colors that change with the seasons.

### The Crystalline Peaks

The crystalline peaks of Foobar are another mar

In [10]:
cloudwatch.print_recent_logs(log_group_name)

Permissions are correctly set for Amazon Bedrock logs.
-------------------------

{
    "timestamp": "2026-03-22T19:39:26Z",
    "accountId": "258622814741",
    "region": "us-west-2",
    "requestId": "c9f54315-dd64-445f-ba9d-d01851e770fb",
    "operation": "InvokeModel",
    "modelId": "arn:aws:bedrock:us-west-2:258622814741:inference-profile/us.amazon.nova-lite-v1:0",
    "input": {
        "inputContentType": "application/json",
        "inputBodyJson": {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "text": "Write an article about the fictional planet Foobar."
                        }
                    ]
                }
            ],
            "inferenceConfig": {
                "maxTokens": 512,
                "temperature": 0.7,
                "topP": 0.9
            }
        },
        "inputTokenCount": 10
    },
    "output": {
        "outputCo

To review the logs within the AWS console, please use the following link to reference the steps outlined in the video:

In [11]:
from IPython.display import HTML
aws_url = os.environ['AWS_CONSOLE_URL']

In [12]:
HTML(f'<a href="{aws_url}" target="_blank">GO TO AWS CONSOLE</a>')
